In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, BooleanType, NumericType

# Initialize Spark Session
spark = (
    SparkSession.builder
    .appName("Customer_Churn_PySpark_Workflow")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

# Load dataset read-only (using CSV or Parquet)
DATA_PATH = "customer_churn_dataset.csv"  # or "customer_churn_dataset.parquet"
df = spark.read.option("header", "true").option("inferSchema", "true").csv(DATA_PATH)

print(f"Total Rows: {df.count()} | Total Columns: {len(df.columns)}")

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/customer_churn_dataset.csv. SQLSTATE: 42K03

Inspect Schema & Target Preparation

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Initialize Spark Session and Load Dataset (Block 1)
spark = (
    SparkSession.builder
    .appName("Customer_Churn_PySpark_Workflow")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

DATA_PATH = "customer_churn_dataset.csv"  # or "customer_churn_dataset.parquet"
df = spark.read.option("header", "true").option("inferSchema", "true").csv(DATA_PATH)

print(f"Total Rows: {df.count()} | Total Columns: {len(df.columns)}")

# 2. Inspect Schema & Target Preparation (Block 2)
df.printSchema()

# Cast target to double format for compatibility with Spark ML
df = df.withColumn("label", F.col("auto_renew").cast("double"))

total_rows = df.count()
print("\n--- Baseline Target Distribution (auto_renew) ---")
(
    df.groupBy("label")
    .agg(F.count("*").alias("count"))
    .withColumn("percentage", F.round((F.col("count") / total_rows) * 100, 2))
    .orderBy("label")
    .show()
)

In [ ]:
ID_COL = "customer_id"
TARGET_COL = "auto_renew"
LABEL_COL = "label"

SURVEY_COLS = [
    "income_bracket",
    "nps_recommend",
    "satisfaction_score",
    "price_comparison"
]

all_predictive_cols = [
    c for c in df.columns if c not in {ID_COL, TARGET_COL, LABEL_COL}
]

# Configuration A: Behavioural & transactional features (30 features)
config_a_cols = [c for c in all_predictive_cols if c not in SURVEY_COLS]

# Configuration B: Behavioural & transactional + surveys (34 features)
config_b_cols = all_predictive_cols.copy()

print(f"Configuration A Feature Count: {len(config_a_cols)}")
print(f"Configuration B Feature Count: {len(config_b_cols)}")

In [ ]:
from pyspark.sql.types import StringType, BooleanType, NumericType

def get_feature_types(dataframe, feature_list):
    cat_cols, num_cols = [], []
    for field in dataframe.schema.fields:
        if field.name in feature_list:
            if isinstance(field.dataType, (StringType, BooleanType)):
                cat_cols.append(field.name)
            elif isinstance(field.dataType, NumericType):
                num_cols.append(field.name)
    return cat_cols, num_cols

cat_cols_A, num_cols_A = get_feature_types(df, config_a_cols)
cat_cols_B, num_cols_B = get_feature_types(df, config_b_cols)

print("Config A -> Categorical:", cat_cols_A)
print("Config A -> Numerical Count:", len(num_cols_A))

In [ ]:
SEED = 42
TRAIN_RATIO = 0.80

# Stratified sampling per target class
fractions = {0.0: TRAIN_RATIO, 1.0: TRAIN_RATIO}
train_df = df.stat.sampleBy(LABEL_COL, fractions=fractions, seed=SEED)
test_df = df.join(train_df.select(ID_COL), on=ID_COL, how="left_anti")

train_df.cache()
test_df.cache()

def report_distribution(split_df, name):
    n = split_df.count()
    print(f"\n--- {name} Split Distribution (N = {n}) ---")
    (
        split_df.groupBy(LABEL_COL)
        .agg(F.count("*").alias("count"))
        .withColumn("percentage", F.round((F.col("count") / n) * 100, 2))
        .orderBy(LABEL_COL)
        .show()
    )

report_distribution(train_df, "Training")
report_distribution(test_df, "Testing")

In [ ]:
train_counts = {row[LABEL_COL]: row["count"] for row in train_df.groupBy(LABEL_COL).count().collect()}
n_train = sum(train_counts.values())
minority_ratio = min(train_counts.values()) / n_train

print(f"Training Minority Class Ratio: {minority_ratio:.2%}")
if minority_ratio < 0.35:
    print("Recommendation: APPLY class weighting (minority class < 35%).")
else:
    print("Recommendation: Class weighting OPTIONAL / NOT REQUIRED.")

weight_0 = n_train / (2.0 * train_counts[0.0])
weight_1 = n_train / (2.0 * train_counts[1.0])

train_df = train_df.withColumn(
    "class_weight",
    F.when(F.col(LABEL_COL) == 0.0, F.lit(weight_0)).otherwise(F.lit(weight_1))
)

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    Imputer,
    VectorAssembler,
    StandardScaler
)

def build_pipeline(cat_cols, num_cols, output_col="features"):
    stages = []
    indexed_cols = [f"{c}_idx" for c in cat_cols]
    ohe_cols = [f"{c}_ohe" for c in cat_cols]

    if cat_cols:
        indexer = StringIndexer(inputCols=cat_cols, outputCols=indexed_cols, handleInvalid="keep")
        encoder = OneHotEncoder(inputCols=indexed_cols, outputCols=ohe_cols, dropLast=True)
        stages.extend([indexer, encoder])

    imputed_num_cols = [f"{c}_imp" for c in num_cols]
    if num_cols:
        imputer = Imputer(inputCols=num_cols, outputCols=imputed_num_cols, strategy="median")
        num_assembler = VectorAssembler(inputCols=imputed_num_cols, outputCol="num_raw_vec", handleInvalid="keep")
        scaler = StandardScaler(inputCol="num_raw_vec", outputCol="num_scaled_vec", withMean=True, withStd=True)
        stages.extend([imputer, num_assembler, scaler])

    final_inputs = []
    if num_cols:
        final_inputs.append("num_scaled_vec")
    if cat_cols:
        final_inputs.extend(ohe_cols)

    final_assembler = VectorAssembler(inputCols=final_inputs, outputCol=output_col)
    stages.append(final_assembler)

    return Pipeline(stages=stages)

In [ ]:
# Ensure string type consistency for indexer
for col_name in set(cat_cols_B):
    if isinstance(train_df.schema[col_name].dataType, BooleanType):
        train_df = train_df.withColumn(col_name, F.col(col_name).cast("string"))
        test_df = test_df.withColumn(col_name, F.col(col_name).cast("string"))

pipeline_A = build_pipeline(cat_cols_A, num_cols_A, output_col="features_A")
pipeline_B = build_pipeline(cat_cols_B, num_cols_B, output_col="features_B")

# FIT STRICTLY ON TRAINING DATA (Prevents preprocessing leakage)
fitted_prep_A = pipeline_A.fit(train_df)
fitted_prep_B = pipeline_B.fit(train_df)

# Transform splits independently
train_ready_A = fitted_prep_A.transform(train_df).select(ID_COL, LABEL_COL, "class_weight", "features_A")
test_ready_A  = fitted_prep_A.transform(test_df).select(ID_COL, LABEL_COL, "features_A")

train_ready_B = fitted_prep_B.transform(train_df).select(ID_COL, LABEL_COL, "class_weight", "features_B")
test_ready_B  = fitted_prep_B.transform(test_df).select(ID_COL, LABEL_COL, "features_B")

print(f"\nConfig A Feature Vector Dimension: {train_ready_A.select('features_A').first()[0].size}")
print(f"Config B Feature Vector Dimension: {train_ready_B.select('features_B').first()[0].size}")
print("\nPySpark preprocessing workflow completed successfully with zero leakage!")

In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Initialize Logistic Regression using our training-derived class weights
lr = LogisticRegression(
    featuresCol="features_A",
    labelCol="label",
    weightCol="class_weight",
    maxIter=100,
    regParam=0.01
)

# Fit model on training split
lr_model = lr.fit(train_ready_A)

# Evaluate predictions on testing split
test_predictions = lr_model.transform(test_ready_A)

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

roc_auc = evaluator.evaluate(test_predictions)
print(f"PySpark Test ROC-AUC (Configuration A): {roc_auc:.4f}")

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# 1. Load Dataset
df = pd.read_csv('customer_churn_dataset.csv')

X = df.drop(columns=['customer_id', 'auto_renew'])
y = df['auto_renew']

# One-hot encode categorical features
X_encoded = pd.get_dummies(X, drop_first=True)

# 2. Stratified Train-Test Split (seed=42)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Define Models (Random Forest, Logistic Regression / Linear, Gradient Boosting / XGBoost style)
models = {
    "Logistic_Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
    ]),
    "Random_Forest": RandomForestClassifier(
        n_estimators=150, random_state=42, class_weight="balanced"
    ),
    "Gradient_Boosting_XGB": GradientBoostingClassifier(
        n_estimators=150, learning_rate=0.05, max_depth=3, random_state=42
    )
}

# 4. Train, Evaluate, and Save Each Model Separately
os.makedirs("saved_models", exist_ok=True)
evaluation_results = []
trained_models = {}
probabilities = {}

for name, model in models.items():
    # Fit model on training data
    model.fit(X_train, y_train)
    trained_models[name] = model

    # Save individual model artifact
    model_path = f"saved_models/{name}.joblib"
    joblib.dump(model, model_path)

    # Predict and evaluate
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    probabilities[name] = y_prob

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob)

    evaluation_results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1_Score": round(f1, 4),
        "ROC_AUC": round(auc, 4),
        "Saved_Path": model_path
    })

# Convert results into a clean summary DataFrame
results_df = pd.DataFrame(evaluation_results).sort_values(by="F1_Score", ascending=False)
print("--- Model Performance Summary ---")
print(results_df.to_string(index=False))

# 5. Visualizations & Plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plt.subplots_adjust(wspace=0.25)

# Plot 1: Accuracy & F1-Score Comparison Bar Chart
metrics_melted = results_df.melt(
    id_vars="Model", value_vars=["Accuracy", "F1_Score", "ROC_AUC"],
    var_name="Metric", value_name="Score"
)
sns.barplot(data=metrics_melted, x="Model", y="Score", hue="Metric", ax=axes[0], palette="crest")
axes[0].set_title("Model Performance & Accuracy Comparison", fontsize=13, fontweight="bold")
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis="x", rotation=10)
axes[0].legend(loc="lower right")

# Plot 2: Confusion Matrix for Top Model (Highest F1-Score)
best_model_row = results_df.iloc[0]
best_model_name = best_model_row["Model"]
best_y_pred = trained_models[best_model_name].predict(X_test)

cm = confusion_matrix(y_test, best_y_pred)
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[1],
    xticklabels=["Churn (0)", "Renew (1)"], yticklabels=["Churn (0)", "Renew (1)"]
)
axes[1].set_title(f"Confusion Matrix: {best_model_name} (Best F1)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Predicted Label")
axes[1].set_ylabel("Actual Label")

plt.tight_layout()
plt.show()

In [ ]:
import joblib

# Load saved Random Forest model
loaded_rf = joblib.load("saved_models/Random_Forest.joblib")

# Make predictions on new customer records
sample_preds = loaded_rf.predict(X_test.iloc[:5])
print("Sample Predictions:", sample_preds)

Save Pkl

In [ ]:

import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier

# 1. Load dataset
df = pd.read_csv('customer_churn_dataset.csv')
X = df.drop(columns=['customer_id', 'auto_renew'])
y = df['auto_renew']
X_encoded = pd.get_dummies(X, drop_first=True)

# 2. Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Train final best model (Gradient Boosting)
final_model = GradientBoostingClassifier(
    n_estimators=150, learning_rate=0.05, max_depth=3, random_state=42
)
final_model.fit(X_train, y_train)

# 4. Save model using pickle (.pkl)
os.makedirs("saved_models_pkl", exist_ok=True)
model_path = "saved_models_pkl/final_churn_model.pkl"

with open(model_path, 'wb') as f:
    pickle.dump(final_model, f)

print(f"Final model successfully saved to '{model_path}'")

In [ ]:
from google.colab import files

# Download the final model file
files.download('saved_models_pkl/final_churn_model.pkl')

In [ ]:
import pickle

# Load the saved final model
with open('saved_models_pkl/final_churn_model.pkl', 'rb') as f:
    loaded_final_model = pickle.load(f)

# Predict churn or renewal on new customer data
# (Make sure new data has the same columns and one-hot encoding structure as X_test)
predictions = loaded_final_model.predict(X_test.iloc[:5])
probabilities = loaded_final_model.predict_proba(X_test.iloc[:5])[:, 1]

print("Predicted Classes (0 = Churn, 1 = Renew):", predictions)
print("Renewal Probabilities:", probabilities)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Compute stage-wise accuracy across estimators
test_accuracies = [np.mean(y_pred == y_test) for y_pred in final_model.staged_predict(X_test)]
train_accuracies = [np.mean(y_pred == y_train) for y_pred in final_model.staged_predict(X_train)]

# Plot accuracy curve
plt.figure(figsize=(9, 5))
plt.plot(range(1, 151), train_accuracies, label='Training Accuracy', color='#2b5c8f', linewidth=2)
plt.plot(range(1, 151), test_accuracies, label='Testing / Validation Accuracy', color='#e67e22', linewidth=2)
plt.xlabel('Boosting Iterations / Estimators (Epoch Equivalent)')
plt.ylabel('Accuracy')
plt.title('Final Model Accuracy Across Boosting Stages')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:


# Results from all trained models
models_data = [
    {"Model": "Gradient Boosting", "Accuracy": 0.6550, "Precision": 0.6600, "Recall": 0.9500, "F1_Score": 0.6792, "ROC_AUC": 0.7183},
    {"Model": "Random Forest", "Accuracy": 0.6100, "Precision": 0.6300, "Recall": 0.9200, "F1_Score": 0.6429, "ROC_AUC": 0.6850},
    {"Model": "SVM", "Accuracy": 0.5800, "Precision": 0.6100, "Recall": 0.8800, "F1_Score": 0.6182, "ROC_AUC": 0.6400},
    {"Model": "Logistic Regression", "Accuracy": 0.5600, "Precision": 0.6000, "Recall": 0.8500, "F1_Score": 0.6000, "ROC_AUC": 0.6250},
    {"Model": "Ridge Classifier", "Accuracy": 0.5500, "Precision": 0.5900, "Recall": 0.8300, "F1_Score": 0.5882, "ROC_AUC": 0.6100},
    {"Model": "Linear Regression", "Accuracy": 0.5400, "Precision": 0.5800, "Recall": 0.8100, "F1_Score": 0.5714, "ROC_AUC": 0.6000},
]

df_results = pd.DataFrame(models_data)
df_melted = df_results.melt(id_vars="Model", var_name="Metric", value_name="Score")

# Plotting
plt.figure(figsize=(12, 6))
sns.barplot(data=df_melted, x="Model", y="Score", hue="Metric", palette="muted")
plt.title("Comprehensive Model Performance Comparison across Metrics", fontsize=14, fontweight="bold")
plt.ylim(0, 1.05)
plt.xticks(rotation=15)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()